---
# Part 3 — Recommendation System

## Collaborative Filtering with ALS (Alternating Least Squares)

**Collaborative filtering** is the standard approach for recommendation systems. The core idea is that *users who agreed in the past tend to agree again in the future* — we fill in the missing entries of the user-item ratings matrix by leveraging patterns across all users.

Spark ML supports **model-based collaborative filtering**, where users and movies are each represented by a small set of **latent factors** (hidden features). The model learns these factors from observed ratings and uses them to predict unobserved ones.

### ALS Algorithm

Spark MLlib uses **Alternating Least Squares (ALS)** to factorize the ratings matrix **R** into two lower-rank matrices:

$$R \approx U \times V^T$$

- **U** — User latent factor matrix  
- **V** — Item (movie) latent factor matrix  

ALS alternates between fixing **U** and solving for **V**, then fixing **V** and solving for **U**, until convergence. This alternation makes the otherwise non-convex problem solvable as a series of least-squares problems, each of which is highly parallelizable on Spark.

📖 *See: [Spark MLlib Collaborative Filtering Docs](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html)*

In [1]:
# give access to google colab to have access to your google drive, in order to be able to read the datasets
#from google.colab import drive
# drive.mount('/content/gdrive')
# google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"
from pathlib import Path
#import pandas as pd

# Notebook location (usually your project folder)
project_dir = Path.cwd()

# Data folder one level above project folder
data_dir = project_dir.parent / "data"   # change "data" if your folder name is different

print("Project dir:", project_dir)
print("Data dir:", data_dir)
print("Exists:", data_dir.exists())

# Optional: see files inside data folder
if data_dir.exists():
    for p in data_dir.iterdir():
        print(p.name)

Project dir: c:\Users\alexa\Documents\UCY files\Data Science\DSC511\DSC511 Project\DSC511-MovieLens-Project
Data dir: c:\Users\alexa\Documents\UCY files\Data Science\DSC511\DSC511 Project\data
Exists: True
links.csv
movies.csv
ratings.csv


In [11]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install seaborn

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 14.7 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 45.3 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.7 MB 16.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 34.0 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------

In [6]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql.functions import *
# from pyspark.sql.functions import col, from_unixtime, to_timestamp
#from pyspark.sql.functions import explode, split, col

spark = SparkSession.builder.appName("DSC_511_PROJECT").master("local[*]").getOrCreate()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Read the ratings and movies data into Spark DataFrames
ratings_df = spark.read.csv(data_dir / 'ratings.csv', header=True, inferSchema=True)
movies_df = spark.read.csv(data_dir / 'movies.csv', header=True, inferSchema=True)

In [ ]:
# Pre processing steps of main analysis
from itertools import count


ratings_clean = ratings_df.withColumn("ts", to_timestamp(from_unixtime(col("timestamp"))))
# merging the dataframes
movies_ratings_merged = ratings_clean.join(movies_df, on="movieId", how="inner")
# removing the row with the genre that is a title
bad_id = movies_df.filter(col("genres").contains("We're Comin'")).select("movieId").first()[0]
#print("Corrupted movieId:", bad_id)
# Drop from all 3 dataframes
movies_df             = movies_df.filter(col("movieId") != bad_id)
ratings_df            = ratings_df.filter(col("movieId") != bad_id)
movies_ratings_merged = movies_ratings_merged.filter(col("movieId") != bad_id)

# other RDD/DataFrames created that might be useful for the analysis
#rating_dist = movies_ratings_merged.groupBy("rating").count().orderBy("rating").toPandas()
#movies_with_year = movies_df.withColumn("release_year",regexp_extract(col("title"), r"\((\d{4})\)", 1))
# top 10 most active users
# Get top 10 users by number of ratings given
#top10_users = movies_ratings_merged.groupBy("userId").agg(count("rating").alias("num_ratings"),round(avg("rating"), 2).alias("avg_rating")).orderBy("num_ratings", ascending=False).limit(10)

In [ ]:
# Train / Test split: 80% / 20%
(training_data, test_data) = ratings_df.randomSplit([0.8, 0.2], seed=48)

#print(f"Training samples : {training_data.count():,}")
#print(f"Test samples     : {test_data.count():,}")

### Build and Train the ALS Model

Key parameters:

| Parameter | Value | Meaning |
|---|---|---|
| `rank` | 10 | Number of latent factors (size of U and V matrices) |
| `maxIter` | 10 | ALS iterations |
| `regParam` | 0.1 | Regularization — prevents overfitting |
| `coldStartStrategy` | `"drop"` | Drop users/items in test set not seen during training, to avoid NaN predictions breaking RMSE |


In [ ]:
#from pyspark.ml.recommendation import ALS
# Create an ALS model for collaborative filtering
als = ALS(rank=10,maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId",
          ratingCol="rating", coldStartStrategy="drop")

# Fit the model to the training data
model = als.fit(training_data)

### Model Evaluation

We evaluate using **Root Mean Squared Error (RMSE)** — the standard metric for rating prediction tasks:

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{r}_i - r_i)^2}$$

Since MovieLens ratings are on a 0.5–5.0 scale, a RMSE in the range of 0.8–1.0 is generally considered good.

In [ ]:
# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test_data)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

In [ ]:
# Also compute MAE
evaluator.setMetricName("mae")
mae = evaluator.evaluate(predictions)
print(f"Mean Absolute Error      (MAE) = {mae:.4f}")

## Hyperparameter Tuning — Cross Validation on a 10% Sample

Training the full ALS model on 33M ratings is computationally expensive, making exhaustive 
hyperparameter search on the complete dataset impractical. A standard and widely accepted 
approach in Big Data is to **tune hyperparameters on a representative sample**, then retrain 
the best configuration on the full dataset.

We use a **10% stratified random sample** (~3.4M ratings) which is large enough to produce 
reliable cross-validation estimates while keeping runtime manageable.

### Search Space

| Parameter | Values | Description |
|---|---|---|
| `rank` |  10, 15 | Number of latent factors |
| `regParam` | 0.01, 0.1 | Regularization strength |
| `maxIter` | 5, 10 | Number of ALS iterations |

This gives **8 configurations × 5 folds = 40 ALS fits** total.  
The best configuration is selected based on **mean RMSE across all 5 folds**, 
then the winning hyperparameters are used to retrain on the full dataset.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ── 10% sample for tuning ──────────────────────────────────────────
sample = ratings_df.sample(fraction=0.1, seed=48)
(train_sample, test_sample) = sample.randomSplit([0.8, 0.2], seed=48)
train_sample.persist()
#print(f"Tuning sample size: {train_sample.count():,}")

# ── ALS base estimator ─────────────────────────────────────────────
als_tuning = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

# ── Parameter grid: 2 × 2 × 2 = 8 combinations ───────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(als_tuning.rank,     [ 10, 15])
    .addGrid(als_tuning.regParam, [0.01, 0.1])
    .addGrid(als_tuning.maxIter,  [5, 10])
    .build()
)

cv_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# ── 5-fold Cross Validation ────────────────────────────────────────
cv = CrossValidator(
    estimator=als_tuning,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=5,
    seed=48
)

cv_model = cv.fit(train_sample)
train_sample.unpersist()

# ── Results: mean RMSE per configuration ──────────────────────────
results = []
for params, avg_metric in zip(cv_model.getEstimatorParamMaps(),
                               cv_model.avgMetrics):
    results.append({
        "rank"     : params[als_tuning.rank],
        "regParam" : params[als_tuning.regParam],
        "maxIter"  : params[als_tuning.maxIter],
        "mean_rmse": round(avg_metric, 4)
    })

results_df = pd.DataFrame(results).sort_values("mean_rmse")
print("\nCross-Validation Results (sorted by mean RMSE):")
print(results_df.to_string(index=False))

# ── Best configuration ─────────────────────────────────────────────
best_rank      = cv_model.bestModel.rank
best_reg_param = cv_model.bestModel._java_obj.parent().getRegParam()
best_max_iter  = cv_model.bestModel._java_obj.parent().getMaxIter()
print(f"\n Best rank     : {best_rank}")
print(f"Best regParam : {best_reg_param}")
print(f"Best maxIter  : {best_max_iter}")
print(f"Best mean RMSE: {results_df.iloc[0]['mean_rmse']}")

# ── Retrain best config on FULL training data ──────────────────────
print("\nRetraining best configuration on full dataset...")
als_best = ALS(
    rank=best_rank,
    regParam=best_reg_param,
    maxIter=best_max_iter,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

training_data.persist()
best_model = als_best.fit(training_data)
training_data.unpersist()

# ── Final evaluation on held-out test set ─────────────────────────
best_predictions = best_model.transform(test_data)

cv_evaluator.setMetricName("rmse")
final_rmse = cv_evaluator.evaluate(best_predictions)

cv_evaluator.setMetricName("mae")
final_mae = cv_evaluator.evaluate(best_predictions)

print(f"\nFinal model RMSE on full test set : {final_rmse:.4f}")
print(f"Final model MAE  on full test set : {final_mae:.4f}")

In [ ]:
from pyspark.sql.functions import col, explode
# Generate recommendations for all users
user_recs = model.recommendForAllUsers(10)  # Generate top 10 recommendations for each user

# convert the recommendations to multiple rows per user with one recommendation in each row
user_recs = user_recs.select(col("userId"), explode(col("recommendations")).alias("recommendations"))

# convert the recommendations column from {movieId, rating} to tow columns movieId  and rating
user_recs = user_recs.selectExpr(
    "userId","recommendations.movieId as movieId","recommendations.rating  as rating")

In [ ]:
#Recommendation for a specific User
# We pick a specific user, show what they have already rated, and then show the top-10 recommendations 
TARGET_USER = 4

# Movies already rated by the user
print(f"Movies already rated by User {TARGET_USER}:")
(
    movies_ratings_merged
    .filter(col("userId") == TARGET_USER)
    .select("title", "genres", "rating")
    .orderBy(desc("rating"))
    .show(20, truncate=False)
)

# Top 10 recommendations — only need to join user_recs (10 rows) with movies_df
print(f"Top 10 recommendations for User {TARGET_USER}:")
(
    user_recs
    .filter(col("userId") == TARGET_USER)
    .join(movies_df, on="movieId", how="inner")
    .select("title", "genres", "rating")
    .orderBy(desc("rating"))
    .show(truncate=False)
)
